# Molecule Autoresearch Experiment Analysis

Analysis of autonomous molecule experiments logged to `results_mol.tsv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

candidates = [Path("results_mol.tsv"), Path("results.tsv")]
results_path = next((path for path in candidates if path.exists()), None)
if results_path is None:
    raise FileNotFoundError("Expected results_mol.tsv (or results.tsv) in the notebook directory.")

df = pd.read_csv(results_path, sep="\t")
expected_cols = ["commit", "backend", "val_loss", "memory_gb", "status", "description"]
missing = [col for col in expected_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

df["val_loss"] = pd.to_numeric(df["val_loss"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].fillna("unknown").str.strip().str.upper()
df["backend"] = df["backend"].fillna("unknown").str.strip().str.lower()
df["description"] = df["description"].fillna("").astype(str)
df["experiment"] = range(len(df))

print(f"Loaded: {results_path}")
print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts().sort_index()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = int(counts.get("KEEP", 0))
n_discard = int(counts.get("DISCARD", 0))
n_crash = int(counts.get("CRASH", 0))
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")
print(f"Crash count: {n_crash}")

valid = df[df["status"] != "CRASH"].copy()
if not valid.empty:
    backend_stats = (
        valid.groupby("backend")
        .agg(
            experiments=("commit", "count"),
            kept=("status", lambda s: int((s == "KEEP").sum())),
            best_val_loss=("val_loss", "min"),
            mean_val_loss=("val_loss", "mean"),
            max_memory_gb=("memory_gb", "max"),
        )
        .sort_values("best_val_loss")
    )
    print("\nBackend summary:")
    display(backend_stats)


In [ ]:
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for _, row in kept.iterrows():
    loss = row["val_loss"]
    desc = row["description"].strip()
    print(
        f"  #{int(row['experiment']):3d}  loss={loss:.6f}  "
        f"mem={row['memory_gb']:.1f}GB  backend={row['backend']}  {desc}"
    )

## Validation Loss Over Time

Track how the best kept `val_loss` evolves as experiments progress. The running minimum shows the current frontier.

In [ ]:
valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
if valid.empty:
    raise ValueError("No non-crash experiments to plot.")

baseline_loss = float(valid.loc[0, "val_loss"])
best_loss = float(valid["val_loss"].min())
loss_span = max(valid["val_loss"].max() - valid["val_loss"].min(), 1e-4)
margin = max(loss_span * 0.12, baseline_loss * 0.01)

fig, ax = plt.subplots(figsize=(16, 8))

discarded = valid[valid["status"] == "DISCARD"]
kept_v = valid[valid["status"] == "KEEP"]

ax.scatter(
    discarded["experiment"],
    discarded["val_loss"],
    c="#cccccc",
    s=18,
    alpha=0.6,
    zorder=2,
    label="Discarded",
)
ax.scatter(
    kept_v["experiment"],
    kept_v["val_loss"],
    c="#2ecc71",
    s=56,
    zorder=4,
    label="Kept",
    edgecolors="black",
    linewidths=0.5,
)

running_best = kept_v["val_loss"].cummin()
ax.step(
    kept_v["experiment"],
    running_best,
    where="post",
    color="#27ae60",
    linewidth=2,
    alpha=0.8,
    zorder=3,
    label="Running best",
)

ax.axhline(baseline_loss, color="#34495e", linestyle="--", linewidth=1.2, alpha=0.7, label="Baseline")

for _, row in kept_v.iterrows():
    desc = row["description"].strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(
        desc,
        (row["experiment"], row["val_loss"]),
        textcoords="offset points",
        xytext=(6, 6),
        fontsize=8.0,
        color="#1a7a3a",
        alpha=0.9,
        rotation=28,
        ha="left",
        va="bottom",
    )

n_total = len(df)
n_kept = len(kept_v)
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation Loss (lower is better)", fontsize=12)
ax.set_title(
    f"Molecule Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements",
    fontsize=14,
)
ax.grid(True, alpha=0.2)
ax.legend(loc="upper right", fontsize=9)
ax.set_ylim(valid["val_loss"].min() - margin, valid["val_loss"].max() + margin)

plt.tight_layout()
plt.savefig("progress_mol.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress_mol.png")
print(f"Baseline val_loss: {baseline_loss:.6f}")
print(f"Best val_loss:     {best_loss:.6f}")

## Summary Statistics

In [ ]:
valid = df[df["status"] != "CRASH"].copy()
if valid.empty:
    raise ValueError("No valid experiments available.")

baseline_loss = float(valid.iloc[0]["val_loss"])
best_loss = float(valid["val_loss"].min())
best_row = valid.loc[valid["val_loss"].idxmin()]
improvement = baseline_loss - best_loss

print(f"Baseline val_loss:  {baseline_loss:.6f}")
print(f"Best val_loss:      {best_loss:.6f}")
print(f"Total improvement:  {improvement:+.6f} ({improvement / baseline_loss * 100:.2f}%)")
print(f"Best experiment:    {best_row['description']}")
print(f"Best backend:       {best_row['backend']}")
print()

print("Cumulative effort per kept improvement:")
kept = df[df["status"] == "KEEP"].copy()
for _, row in kept.iterrows():
    desc = row["description"].strip()
    print(
        f"  Experiment #{int(row['experiment']):3d}: "
        f"loss={row['val_loss']:.6f}  backend={row['backend']}  {desc}"
    )

## Top Hits (Kept Experiments by Improvement)

In [ ]:
kept = df[df["status"] == "KEEP"].copy().reset_index(drop=True)
if len(kept) <= 1:
    print("Need at least two kept experiments to compute deltas.")
else:
    kept["prev_val_loss"] = kept["val_loss"].shift(1)
    kept["delta"] = kept["prev_val_loss"] - kept["val_loss"]
    hits = kept.iloc[1:].copy().sort_values("delta", ascending=False)

    print(f"{'Rank':>4}  {'Delta':>10}  {'val_loss':>10}  {'Backend':>8}  Description")
    print("-" * 96)
    for rank, (_, row) in enumerate(hits.iterrows(), 1):
        print(
            f"{rank:4d}  {row['delta']:+.6f}  {row['val_loss']:.6f}  "
            f"{row['backend']:>8}  {row['description']}"
        )

    print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  {'':>8}  TOTAL improvement over baseline")